---
title: VirtualiZarr integration
---

# Virtual Zarr references with async-hdf5

async-hdf5 can create **virtual Zarr references** — chunk manifests that
record byte offsets into remote HDF5 files without reading any array data.
This is the foundation of
[VirtualiZarr](https://github.com/zarr-developers/VirtualiZarr): making
archival data (HDF5, NetCDF4) accessible as Zarr without copying a byte.

## Setup

In [ ]:
import warnings

warnings.filterwarnings("ignore", message="Numcodecs codecs are not in the Zarr")
warnings.filterwarnings(
    "ignore", category=UserWarning, message=".*does not have a Zarr V3 specification.*"
)
warnings.filterwarnings("ignore", category=FutureWarning, module="earthaccess")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="tqdm")

In [ ]:
from urllib.parse import urlparse

import earthaccess
from async_hdf5.store import HTTPStore

auth = earthaccess.login()

results = earthaccess.search_data(
    short_name="ATL06",
    cloud_hosted=True,
    bounding_box=(-55, 60, -30, 85),  # Greenland
    temporal=("2024-01-01", "2024-01-15"),
    count=1,
)

granule = results[0]
https_links = granule.data_links()
url = next(l for l in https_links if l.endswith(".h5"))
print(f"URL: {url}")

token = auth.token["access_token"]

parsed = urlparse(url)
dir_path, _, filename = parsed.path.rpartition("/")
base_url = f"{parsed.scheme}://{parsed.netloc}{dir_path}"

store = HTTPStore.from_url(
    base_url,
    client_options={"default_headers": {"Authorization": f"Bearer {token}"}},
)

## Create virtual references

`open_virtual_hdf5` parses the HDF5 metadata and chunk indices, then builds
a `ManifestStore` — a mapping from Zarr chunk keys to `(url, offset, length)`
triples. No array data is read.

In [ ]:
from async_hdf5.virtualizarr import open_virtual_hdf5

manifest_store = await open_virtual_hdf5(
    filename,
    store=store,
    group="gt1l/land_ice_segments",
    url=url,
)
print(f"Type: {type(manifest_store).__name__}")

## Inspect the manifest

The manifest records where each chunk lives in the original HDF5 file.

In [ ]:
vds = manifest_store.to_virtual_dataset()
vds

In [ ]:
# Show chunk manifests for h_li
if "h_li" in vds:
    manifest = vds["h_li"].data.manifest
    print(f"h_li chunks: {len(manifest)}")
    # Show first few entries
    print(manifest)

## Read through the virtual store

The manifest can be used to read data on demand — the same byte-range
fetch mechanism, but driven by Zarr chunk keys instead of HDF5 internals.

In [ ]:
import xarray as xr

ds = xr.open_dataset(
    filename,
    engine="async_hdf5",
    group="gt1l/land_ice_segments",
    store=store,
)

print(f"h_li shape: {ds['h_li'].shape}")
print(f"First 10 values: {ds['h_li'].values[:10]}")
ds.close()

## The bigger picture

Virtual references are the building block for **connected data products**:

1. **VirtualiZarr** creates virtual Zarr stores from archival files
2. **Icechunk** stores virtual references in a versioned, transactional
   format — git-like snapshots, branches, and time travel
3. When new data arrives, append and commit — no reprocessing
4. Anyone can read the result as a plain Zarr store

The combination means you can build data products that reference petabytes
of archival data without duplicating a single byte — and update them
incrementally as new observations come in.